In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
import re
import gensim
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Conv1D, MaxPooling1D, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pickle


In [ ]:
fake_df = pd.read_csv('ml_assets/dataset/Fake.csv')
true_df = pd.read_csv('ml_assets/dataset/True.csv')

In [ ]:
print(fake_df.shape,true_df.shape)

(23481, 4) (21417, 4)


In [ ]:
fake_df['label'] = 1
true_df['label'] = 0

In [ ]:
df=pd.concat([fake_df,true_df],axis=0)
df.shape

(44898, 5)

In [ ]:
df['text'] = df['title'] + " " + df['text']
df.drop(['title','subject','date'],axis=1,inplace=True)
df.shape

(44898, 2)

In [ ]:
df['text'].isna().sum()

np.int64(0)

In [ ]:
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    # Convert to lowercase
    text = str(text).lower()

    # Remove URLs and emails
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)

    # Remove special characters and digits (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords and lemmatize (keep words > 2 chars)
    tokens = [lemmatizer.lemmatize(word) for word in tokens
              if word not in stop_words and len(word) > 2]

    return ' '.join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
df['text'] = df['text'].apply(preprocess_text)

In [ ]:
df.to_csv('preprocessed/preprocessed_data.csv',index=False)

In [ ]:
X = [text.split() for text in df['text'].tolist()]
y = df['label'].values

In [ ]:
DIM = 100
vector_model = gensim.models.Word2Vec(X, vector_size=DIM, window=10, min_count=1)

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)
X = tokenizer.texts_to_sequences(X)
X = pad_sequences(X, maxlen=7000)

In [ ]:
with open('ml_assets/models/tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
vocab = tokenizer.word_index
vocab_size = len(tokenizer.word_index) + 1

In [ ]:
def create_weight_matrix(vector_model):
  weight_matrix = np.zeros((vocab_size, DIM))

  for word,i in vocab.items():
    weight_matrix[i] = vector_model.wv[word]

  return weight_matrix


In [ ]:
embedding_vectors = create_weight_matrix(vector_model)

In [ ]:
model = Sequential([
    Embedding(
        vocab_size,
        DIM,
        weights=[embedding_vectors],
        input_length=7000,
    ),
    # First LSTM layer
    LSTM(128, return_sequences=True, dropout=0.3),

    # Second LSTM layer
    LSTM(64, dropout=0.3),

    # Dense layers
    Dense(32, activation='relu'),
    Dropout(0.2),

    # Output layer
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │    19,639,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,639,900 (74.92 MB)

 Trainable params: 19,639,900 (74.92 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y)

In [ ]:
model.fit(X_train, y_train, validation_split=0.3, epochs=6)

Epoch 1/6
737/737 ━━━━━━━━━━━━━━━━━━━━ 360s 480ms/step - accuracy: 0.8944 - loss: 0.2545 - val_accuracy: 0.9687 - val_loss: 0.0895
Epoch 2/6
737/737 ━━━━━━━━━━━━━━━━━━━━ 358s 486ms/step - accuracy: 0.9654 - loss: 0.0939 - val_accuracy: 0.9970 - val_loss: 0.0102
Epoch 3/6
737/737 ━━━━━━━━━━━━━━━━━━━━ 379s 482ms/step - accuracy: 0.9967 - loss: 0.0133 - val_accuracy: 0.9989 - val_loss: 0.0032
Epoch 4/6
737/737 ━━━━━━━━━━━━━━━━━━━━ 382s 482ms/step - accuracy: 0.9984 - loss: 0.0060 - val_accuracy: 0.9986 - val_loss: 0.0056
Epoch 5/6
737/737 ━━━━━━━━━━━━━━━━━━━━ 356s 483ms/step - accuracy: 0.9985 - loss: 0.0061 - val_accuracy: 0.9986 - val_loss: 0.0042
Epoch 6/6
737/737 ━━━━━━━━━━━━━━━━━━━━ 393s 533ms/step - accuracy: 0.9980 - loss: 0.0075 - val_accuracy: 0.9986 - val_loss: 0.0034


In [ ]:
y_pred = (model.predict(X_test) >=0.5).astype(int)

351/351 ━━━━━━━━━━━━━━━━━━━━ 70s 199ms/step


In [ ]:
print("Accuracy Score: ",accuracy_score(y_test,y_pred)*100)
print(classification_report(y_test,y_pred))

Accuracy Score:  99.79510022271715
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5366
           1       1.00      1.00      1.00      5859

    accuracy                           1.00     11225
   macro avg       1.00      1.00      1.00     11225
weighted avg       1.00      1.00      1.00     11225



In [ ]:
model.save('/ml_assets/models/isot_double_lstm_layer.h5')